# **Langkah 1: Pre-Processing Data**

###**1.1 Load Dataset**

In [ ]:
df = pd.read_parquet(DATASET_PATH)

print(f"Dataset berhasil diload!")
print(f"  - Total rows: {df.shape[0]:,}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset berhasil diload!
  - Total rows: 685,671
  - Total columns: 95
  - Memory usage: 3994.50 MB


###**1.2 Inspeksi Awal**

1.2.1 Cek Kolom dan Tipe Data

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 685671 entries, 0 to 685670
Data columns (total 95 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   device_name                           685671 non-null  object 
 1   device_mac                            685671 non-null  object 
 2   label_full                            685671 non-null  object 
 3   label1                                685671 non-null  object 
 4   label2                                685671 non-null  object 
 5   label3                                685671 non-null  object 
 6   label4                                685671 non-null  object 
 7   timestamp                             685671 non-null  object 
 8   timestamp_start                       685671 non-null  object 
 9   timestamp_end                         685671 non-null  object 
 10  log_data-ranges_avg                   685671 non-null  float64
 11  

1.2.2 Distribusi Kelas (Label2 - Original)

In [ ]:
label_col = 'label2'
class_counts_original = df[label_col].value_counts().sort_index()

print(f"\nDistribusi Kelas (8 Kategori Original):")
for label in class_counts_original.index:
    count = class_counts_original[label]
    pct = (count / len(df)) * 100
    print(f"  {label:12s}: {count:7,} ({pct:5.2f}%)")

print(f"\nTotal samples: {len(df):,}")
print(f"Jumlah kelas: {df[label_col].nunique()}")


Distribusi Kelas (8 Kategori Original):
  benign      : 400,672 (58.44%)
  bruteforce  :   6,016 ( 0.88%)
  ddos        :  56,692 ( 8.27%)
  dos         :  57,736 ( 8.42%)
  malware     :  24,177 ( 3.53%)
  mitm        :  25,490 ( 3.72%)
  recon       : 105,848 (15.44%)
  web         :   9,040 ( 1.32%)

Total samples: 685,671
Jumlah kelas: 8


In [ ]:
print(f"Persentase Kelas:")
class_pct = (class_counts_original / len(df) * 100).round(4)
print(class_pct)

Persentase Kelas:
label2
benign        58.4350
bruteforce     0.8774
ddos           8.2681
dos            8.4204
malware        3.5260
mitm           3.7175
recon         15.4371
web            1.3184
Name: count, dtype: float64


1.2.3 Menghitung Imbalance Ratio

In [ ]:
min_class = class_counts_original.min()
max_class = class_counts_original.max()
imbalance_ratio = max_class / min_class

print(f"Imbalance Ratio: 1:{imbalance_ratio:.0f}")
print(f"Kelas Mayoritas: {class_counts_original.idxmax()} ({max_class:,} samples)")
print(f"Kelas Minoritas: {class_counts_original.idxmin()} ({min_class:,} samples)")

Imbalance Ratio: 1:67
Kelas Mayoritas: benign (400,672 samples)
Kelas Minoritas: bruteforce (6,016 samples)


1.2.4 Menyimpan Distribusi Kelas

In [ ]:
class_dist_dict = {
    'counts': {str(k): int(v) for k, v in class_counts_original.to_dict().items()},
    'percentages': {str(k): float(v) for k, v in class_pct.to_dict().items()},
    'imbalance_ratio': float(imbalance_ratio),
    'majority_class': str(class_counts_original.idxmax()),
    'minority_class': str(class_counts_original.idxmin())
}

with open(os.path.join(RESULTS_DIR, 'class_distribution.json'), 'w') as f:
    json.dump(class_dist_dict, f, indent=4)

print("Distribusi kelas tersimpan!")

Distribusi kelas tersimpan!


###**1.3 Drop Kolom yang Tidak Diperlukan**

In [ ]:
# Define columns to drop (sama seperti preprocessing sebelumnya)
drop_columns = [
    # Metadata (tidak predictive)
    'device_name',        # Categorical identifier
    'device_mac',         # Unique identifier
    'timestamp',          # String timestamp
    'timestamp_start',    # String timestamp
    'timestamp_end',      # String timestamp

    # Labels (keep label2 only sebagai target)
    'label_full', 'label1', 'label3', 'label4',

    # List columns (pakai count saja, drop list-nya)
    'log_data-types',           # List of data types
    'network_ips_all',          # List of IPs
    'network_ips_dst',          # List of destination IPs
    'network_ips_src',          # List of source IPs
    'network_macs_all',         # List of MACs
    'network_macs_dst',         # List of destination MACs
    'network_macs_src',         # List of source MACs
    'network_ports_all',        # List of ports
    'network_ports_dst',        # List of destination ports
    'network_ports_src',        # List of source ports
    'network_protocols_all',    # List of protocols
    'network_protocols_dst',    # List of destination protocols
    'network_protocols_src'     # List of source protocols
]

# Verify all columns exist before dropping
existing_drop_cols = [col for col in drop_columns if col in df.columns]
missing_drop_cols = [col for col in drop_columns if col not in df.columns]

if missing_drop_cols:
    print(f"Warning: Columns not found in dataset: {missing_drop_cols}")

# Drop columns
df = df.drop(columns=existing_drop_cols)

print(f"Dropped {len(existing_drop_cols)} columns")
print(f"  - Remaining columns: {df.shape[1]}")
print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Tampilkan kolom yang tersisa
print(f"\n  Kolom yang tersisa ({df.shape[1]} kolom):")
print(f"    - label2: 1 kolom (target)")
print(f"    - time_window: 1 kolom (untuk split)")
print(f"    - Features: {df.shape[1] - 2} kolom")

Dropped 22 columns
  - Remaining columns: 73
  - Memory usage: 412.21 MB

  Kolom yang tersisa (73 kolom):
    - label2: 1 kolom (target)
    - time_window: 1 kolom (untuk split)
    - Features: 71 kolom


###**1.4 Konversi Tipe Data (Memory Optimization)**

In [ ]:
initial_memory = df.memory_usage(deep=True).sum() / 1024**2

# Downcast integer columns (kecuali time_window dan label2)
int_cols = df.select_dtypes(include=['int64']).columns.tolist()
exclude_int = ['time_window']  # label2 adalah string, jadi tidak perlu exclude

for col in int_cols:
    if col not in exclude_int:
        # Check if values fit in int32
        col_min = df[col].min()
        col_max = df[col].max()
        if col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
            df[col] = df[col].astype('int32')
        else:
            print(f"   {col}: Range too large for int32, keeping int64")

# Downcast float columns
float_cols = df.select_dtypes(include=['float64']).columns.tolist()
for col in float_cols:
    df[col] = df[col].astype('float32')

final_memory = df.memory_usage(deep=True).sum() / 1024**2
memory_saved = initial_memory - final_memory
memory_reduction = (memory_saved / initial_memory) * 100

print(f"Memory optimization complete!")
print(f"  - Initial memory: {initial_memory:.2f} MB")
print(f"  - Final memory: {final_memory:.2f} MB")
print(f"  - Saved: {memory_saved:.2f} MB ({memory_reduction:.1f}% reduction)")

gc.collect()  # Force garbage collection

Memory optimization complete!
  - Initial memory: 412.21 MB
  - Final memory: 226.50 MB
  - Saved: 185.71 MB (45.1% reduction)


0

###**1.5 Cek Kualitas Data & Cleaning**

1.5.1 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
missing_df = pd.DataFrame({
    'Jumlah_Missing': missing,
    'Persentase_Missing': missing_pct
})
missing_df = missing_df[missing_df['Jumlah_Missing'] > 0].sort_values('Jumlah_Missing', ascending=False)

if len(missing_df) > 0:
    print(" Kolom dengan missing values:")
    print(missing_df)

    # Handle missing values
    print("\n  Menangani missing values...")
    for col in missing_df.index:
        if missing_df.loc[col, 'Persentase_Missing'] < 1.0:
            # Hapus baris dengan missing < 1%
            df = df.dropna(subset=[col])
            print(f"    - Baris dengan missing values di '{col}' dihapus")
        else:
            # Fill dengan median/mode untuk missing >= 1%
            if df[col].dtype in ['float32', 'float64', 'int32', 'int64', 'Int32']:
                df[col].fillna(df[col].median(), inplace=True)
                print(f"    - Missing values di '{col}' diisi dengan median")
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f"    - Missing values di '{col}' diisi dengan modus")
else:
    print("Tidak ada missing values!")

Tidak ada missing values!


1.5.2 Infinite Values

In [ ]:
# Pilih kolom numerik saja
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

inf_counts = {}
for col in numeric_cols:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        inf_counts[col] = inf_count

if inf_counts:
    print(" Kolom dengan nilai infinite:")
    for col, count in inf_counts.items():
        print(f"    - {col}: {count:,}")

    print("\n  Mengganti nilai infinite dengan max/min kolom...")
    for col in inf_counts.keys():
        # Mengganti nilai np.inf dengan nilai maksimum yang valid di kolom tersebut
        df[col] = df[col].replace([np.inf], df[col][~np.isinf(df[col])].max())
        # Mengganti nilai -np.inf dengan nilai minimum yang valid di kolom tersebut
        df[col] = df[col].replace([-np.inf], df[col][~np.isinf(df[col])].min())
        print(f"    - Nilai infinite di '{col}' diperbaiki")
else:
    print(" Tidak ada nilai infinite!")

 Tidak ada nilai infinite!


1.5.3 Remove All-Zero Rows

In [ ]:
# Kolom fitur (exclude label2 dan time_window)
feature_cols = [col for col in df.columns if col not in ['label2', 'time_window']]

# Cek baris yang semua fiturnya 0
all_zero_mask = (df[feature_cols] == 0).all(axis=1)
zero_count = all_zero_mask.sum()

print(f"  Ditemukan {zero_count:,} all-zero rows ({(zero_count/len(df)*100):.2f}%)")

if zero_count > 0:
    df = df[~all_zero_mask].copy()
    print(f"  Removed {zero_count:,} invalid rows")
    print(f"  Remaining: {len(df):,} rows")
else:
    print("  Tidak ada all-zero rows")

  Ditemukan 136,798 all-zero rows (19.95%)
  Removed 136,798 invalid rows
  Remaining: 548,873 rows


##### 1.5.4 Remove Exact Duplicates (same time_window)

In [ ]:
# Kolom untuk cek duplikat (semua kolom kecuali label2)
feature_cols_with_tw = [col for col in df.columns if col not in ['label2']]

size_before = len(df)
df = df.drop_duplicates(subset=feature_cols_with_tw, keep='first')
exact_dup_removed = size_before - len(df)

print(f"  Removed {exact_dup_removed:,} exact duplicates")
print(f"  Remaining: {len(df):,} rows")

  Removed 2,231 exact duplicates
  Remaining: 546,642 rows


#####1.5.5 Remove Temporal Duplicates (different time_window)

In [ ]:
size_before_temporal = len(df)

# Get feature columns only (exclude label2 and time_window)
feature_cols = [col for col in df.columns if col not in ['label2', 'time_window']]

print(f"  Checking duplicates across {len(feature_cols)} features...")
print(f"  (This removes rows with identical features regardless of time_window)")

# Remove duplicates based on features only
df = df.drop_duplicates(subset=feature_cols, keep='first')
temporal_dup_removed = size_before_temporal - len(df)

print(f"  Removed {temporal_dup_removed:,} temporal duplicates")
print(f"  Remaining: {len(df):,} rows")

  Checking duplicates across 71 features...
  (This removes rows with identical features regardless of time_window)
  Removed 115,049 temporal duplicates
  Remaining: 431,593 rows


1.5.6 Distribusi Kelas Setelah Cleaning

In [ ]:
class_counts_clean = df['label2'].value_counts().sort_index()

print(f"\nDistribusi Kelas Setelah Cleaning:")
for label in class_counts_clean.index:
    count = class_counts_clean[label]
    pct = (count / len(df)) * 100
    print(f"  {label:12s}: {count:7,} ({pct:5.2f}%)")

print(f"\n[Summary Cleaning]")
print(f"  Original size: 685,671")
print(f"  After cleaning: {len(df):,}")
print(f"  Total removed: {685671 - len(df):,} ({((685671 - len(df))/685671*100):.1f}%)")


Distribusi Kelas Setelah Cleaning:
  benign      : 179,312 (41.55%)
  bruteforce  :   4,626 ( 1.07%)
  ddos        :  51,841 (12.01%)
  dos         :  55,469 (12.85%)
  malware     :  22,968 ( 5.32%)
  mitm        :  23,549 ( 5.46%)
  recon       :  86,163 (19.96%)
  web         :   7,665 ( 1.78%)

[Summary Cleaning]
  Original size: 685,671
  After cleaning: 431,593
  Total removed: 254,078 (37.1%)


###**1.6 Definisi Mapping Kategori (8 Kelas)**

In [ ]:
# Mapping dari label2 CICIIoT2025 ke 8 kategori standar
CATEGORY_MAPPING = {
    'Benign': ['benign'],
    'Recon': ['recon'],
    'DoS': ['dos'],
    'DDoS': ['ddos'],
    'MITM/Spoofing': ['mitm'],          # MITM = Spoofing
    'Malware/Mirai': ['malware'],          # Malware = Mirai
    'Web_Based': ['web'],
    'Brute_Force': ['bruteforce']
}

# Fungsi untuk mapping label2 ke kategori
def map_to_category(label):
    """Map label2 ke kategori standar"""
    for category, labels in CATEGORY_MAPPING.items():
        if label in labels:
            return category
    return 'Unknown'  # Fallback jika ada label yang tidak terdaftar

# Terapkan mapping ke dataframe
print("  Menerapkan kategori grouping...")
df['Category'] = df['label2'].apply(map_to_category)

# Cek hasil mapping
print(f"\n Distribusi Kategori (8 Kelas):")
category_counts = df['Category'].value_counts().sort_index()
for category in category_counts.index:
    count = category_counts[category]
    pct = (count / len(df)) * 100

    # Tampilkan juga original label yang dimapping
    original_labels = CATEGORY_MAPPING.get(category, [])
    print(f"  {category:12s}: {count:7,} ({pct:5.2f}%) <- {original_labels}")

# Hitung imbalance ratio untuk kategori
min_cat = category_counts.min()
max_cat = category_counts.max()
cat_imbalance_ratio = max_cat / min_cat

print(f"\n  Imbalance Ratio (Kategori): 1:{cat_imbalance_ratio:.0f}")
print(f"  Kategori Mayoritas: {category_counts.idxmax()} ({max_cat:,} samples)")
print(f"  Kategori Minoritas: {category_counts.idxmin()} ({min_cat:,} samples)")

# Simpan distribusi kategori
category_dist_dict = {
    'counts': {str(k): int(v) for k, v in category_counts.to_dict().items()},
    'percentages': {str(k): float((v/len(df)*100)) for k, v in category_counts.to_dict().items()},
    'imbalance_ratio': float(cat_imbalance_ratio),
    'majority_class': str(category_counts.idxmax()),
    'minority_class': str(category_counts.idxmin()),
    'mapping': CATEGORY_MAPPING
}

with open(os.path.join(RESULTS_DIR, 'category_distribution.json'), 'w') as f:
    json.dump(category_dist_dict, f, indent=4)

print(f"\n Distribusi kategori tersimpan di: {RESULTS_DIR}")

  Menerapkan kategori grouping...

 Distribusi Kategori (8 Kelas):
  Benign      : 179,312 (41.55%) <- ['benign']
  Brute_Force :   4,626 ( 1.07%) <- ['bruteforce']
  DDoS        :  51,841 (12.01%) <- ['ddos']
  DoS         :  55,469 (12.85%) <- ['dos']
  MITM/Spoofing:  23,549 ( 5.46%) <- ['mitm']
  Malware/Mirai:  22,968 ( 5.32%) <- ['malware']
  Recon       :  86,163 (19.96%) <- ['recon']
  Web_Based   :   7,665 ( 1.78%) <- ['web']

  Imbalance Ratio (Kategori): 1:39
  Kategori Mayoritas: Benign (179,312 samples)
  Kategori Minoritas: Brute_Force (4,626 samples)

 Distribusi kategori tersimpan di: /content/drive/My Drive/CICIIoT2025/Baseline8Multiclass/Results/


###**1.7 Pemisahan Fitur dengan Label**

In [ ]:
# Target column adalah 'Category' (bukan label2)
label_col = 'Category'

# Drop kolom yang tidak dipakai sebagai fitur
X = df.drop(columns=['label2', 'Category', 'time_window'])
y = df[label_col]

# Simpan time_window untuk split nanti
time_windows = df['time_window']

print(f" Pemisahan fitur dan label selesai!")
print(f"  - Ukuran fitur (X): {X.shape}")
print(f"  - Ukuran label (y): {y.shape}")
print(f"  - Jumlah fitur: {X.shape[1]}")
print(f"  - Jumlah kategori: {y.nunique()}")

# Simpan nama fitur
feature_names = X.columns.tolist()
print(f"\n  Feature names ({len(feature_names)} features):")
print(f"    {feature_names[:5]} ... {feature_names[-3:]}")

 Pemisahan fitur dan label selesai!
  - Ukuran fitur (X): (431593, 71)
  - Ukuran label (y): (431593,)
  - Jumlah fitur: 71
  - Jumlah kategori: 8

  Feature names (71 features):
    ['log_data-ranges_avg', 'log_data-ranges_max', 'log_data-ranges_min', 'log_data-ranges_std_deviation', 'log_data-types_count'] ... ['network_window-size_max', 'network_window-size_min', 'network_window-size_std_deviation']


###**1.8 Label Encoding**

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n Mapping Label Encoding (8 Kategori):")
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

for label_name, label_encoded in sorted(label_mapping.items(), key=lambda x: x[1]):
    count = (y == label_name).sum()
    pct = (count / len(y)) * 100
    print(f"  {label_encoded}: {label_name:12s} ({count:7,} samples, {pct:5.2f}%)")

# Simpan label encoder
with open(os.path.join(PROCESSED_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

# Simpan mapping
label_mapping_readable = {str(k): int(v) for k, v in label_mapping.items()}
with open(os.path.join(RESULTS_DIR, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping_readable, f, indent=4)

# Simpan juga category mapping
with open(os.path.join(RESULTS_DIR, 'category_mapping.json'), 'w') as f:
    json.dump(CATEGORY_MAPPING, f, indent=4)

print(f"\n Label encoder tersimpan!")


 Mapping Label Encoding (8 Kategori):
  0: Benign       (179,312 samples, 41.55%)
  1: Brute_Force  (  4,626 samples,  1.07%)
  2: DDoS         ( 51,841 samples, 12.01%)
  3: DoS          ( 55,469 samples, 12.85%)
  4: MITM/Spoofing ( 23,549 samples,  5.46%)
  5: Malware/Mirai ( 22,968 samples,  5.32%)
  6: Recon        ( 86,163 samples, 19.96%)
  7: Web_Based    (  7,665 samples,  1.78%)

 Label encoder tersimpan!


###**1.9 Split Data (Time-Based)**

In [ ]:
# Define time windows untuk train dan test
train_windows = [1, 2, 3, 4, 5, 6, 7, 8]
test_windows = [9, 10]

print(f"  Train windows: {train_windows}")
print(f"  Test windows: {test_windows}")

# Split berdasarkan time_window
train_mask = time_windows.isin(train_windows)
test_mask = time_windows.isin(test_windows)

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y_encoded[train_mask]
y_test = y_encoded[test_mask]

print(f"\n Split complete!")
print(f"  - Train set: {X_train.shape[0]:,} rows ({(X_train.shape[0]/len(X)*100):.1f}%)")
print(f"  - Test set: {X_test.shape[0]:,} rows ({(X_test.shape[0]/len(X)*100):.1f}%)")

# Distribusi kelas di train set
print(f"\n  Distribusi kelas di train set:")
train_dist = pd.Series(y_train).value_counts().sort_index()
for idx in train_dist.index:
    class_name = le.inverse_transform([idx])[0]
    count = train_dist[idx]
    pct = (count / len(y_train)) * 100
    print(f"    {idx} ({class_name:12s}): {count:7,} ({pct:5.2f}%)")

# Distribusi kelas di test set
print(f"\n  Distribusi kelas di test set:")
test_dist = pd.Series(y_test).value_counts().sort_index()
for idx in test_dist.index:
    class_name = le.inverse_transform([idx])[0]
    count = test_dist[idx]
    pct = (count / len(y_test)) * 100
    print(f"    {idx} ({class_name:12s}): {count:7,} ({pct:5.2f}%)")

# Cleanup untuk hemat memory
del df, time_windows, train_mask, test_mask
gc.collect()

print(f"\n Memory cleanup selesai!")

  Train windows: [1, 2, 3, 4, 5, 6, 7, 8]
  Test windows: [9, 10]

 Split complete!
  - Train set: 386,274 rows (89.5%)
  - Test set: 45,319 rows (10.5%)

  Distribusi kelas di train set:
    0 (Benign      ): 163,578 (42.35%)
    1 (Brute_Force ):   4,055 ( 1.05%)
    2 (DDoS        ):  45,744 (11.84%)
    3 (DoS         ):  49,107 (12.71%)
    4 (MITM/Spoofing):  20,799 ( 5.38%)
    5 (Malware/Mirai):  20,195 ( 5.23%)
    6 (Recon       ):  76,067 (19.69%)
    7 (Web_Based   ):   6,729 ( 1.74%)

  Distribusi kelas di test set:
    0 (Benign      ):  15,734 (34.72%)
    1 (Brute_Force ):     571 ( 1.26%)
    2 (DDoS        ):   6,097 (13.45%)
    3 (DoS         ):   6,362 (14.04%)
    4 (MITM/Spoofing):   2,750 ( 6.07%)
    5 (Malware/Mirai):   2,773 ( 6.12%)
    6 (Recon       ):  10,096 (22.28%)
    7 (Web_Based   ):     936 ( 2.07%)

 Memory cleanup selesai!


###**1.10 Verifikasi No Data Leakage**

In [ ]:
# Hash semua fitur untuk cek overlap
print("  Checking for data leakage...")

test_hash = X_test.apply(lambda x: hash(tuple(x)), axis=1)
train_hash = X_train.apply(lambda x: hash(tuple(x)), axis=1)

overlap = test_hash.isin(train_hash).sum()
overlap_pct = (overlap / len(X_test)) * 100

print(f"\n  Test rows with identical features in train: {overlap} / {len(X_test)} ({overlap_pct:.2f}%)")

if overlap == 0:
    print(f"\n  SUCCESS! NO DATA LEAKAGE DETECTED!")
    print(f"  Time-based split is VALID for unbiased evaluation!")
else:
    print(f"\n  WARNING: {overlap_pct:.2f}% leakage detected!")
    print(f"  Need further investigation...")

  Checking for data leakage...

  Test rows with identical features in train: 0 / 45319 (0.00%)

  SUCCESS! NO DATA LEAKAGE DETECTED!
  Time-based split is VALID for unbiased evaluation!


###**1.11 Normalisasi Fitur (StandardScaler)**

In [ ]:
scaler = StandardScaler()

# Fit pada train, transform train & test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f" Fitur dinormalisasi menggunakan StandardScaler")
print(f"  - Mean dari scaled train set: {X_train_scaled.mean():.10f}")
print(f"  - Std dari scaled train set: {X_train_scaled.std():.10f}")

# Validasi detail scaling
print(f"\n[Validasi Detail Scaling]")

# Cek mean per fitur (seharusnya semua mendekati 0)
means_per_feature = X_train_scaled.mean(axis=0)
print(f"  Mean per fitur:")
print(f"    - Min: {means_per_feature.min():.10f}")
print(f"    - Max: {means_per_feature.max():.10f}")
print(f"    - Rata-rata: {means_per_feature.mean():.10f}")

# Cek std per fitur (seharusnya semua mendekati 1)
stds_per_feature = X_train_scaled.std(axis=0)
print(f"  Std per fitur:")
print(f"    - Min: {stds_per_feature.min():.10f}")
print(f"    - Max: {stds_per_feature.max():.10f}")
print(f"    - Rata-rata: {stds_per_feature.mean():.10f}")

# Cek range data
print(f"  Range data setelah scaling:")
print(f"    - Min value: {X_train_scaled.min():.4f}")
print(f"    - Max value: {X_train_scaled.max():.4f}")

# Cek beberapa sample
print(f"\n  Sample data (5 baris pertama, 5 kolom pertama):")
print(X_train_scaled[:5, :5])

# Simpan scaler
with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"\n Scaler tersimpan!")

 Fitur dinormalisasi menggunakan StandardScaler
  - Mean dari scaled train set: 0.0000000000
  - Std dari scaled train set: 1.0000000000

[Validasi Detail Scaling]
  Mean per fitur:
    - Min: -0.0000000000
    - Max: 0.0000000000
    - Rata-rata: 0.0000000000
  Std per fitur:
    - Min: 1.0000000000
    - Max: 1.0000000000
    - Rata-rata: 1.0000000000
  Range data setelah scaling:
    - Min value: -3.9697
    - Max value: 59.5198

  Sample data (5 baris pertama, 5 kolom pertama):
[[-0.35513275 -0.36028245 -0.34969588 -0.0950422  -0.68966356]
 [-0.35513275 -0.36028245 -0.34969588 -0.0950422  -0.68966356]
 [-0.35513275 -0.36028245 -0.34969588 -0.0950422  -0.68966356]
 [-0.35513275 -0.36028245 -0.34969588 -0.0950422  -0.68966356]
 [-0.35513275 -0.36028245 -0.34969588 -0.0950422  -0.68966356]]

 Scaler tersimpan!


###**1.12 Menyimpan Data Terproses**

In [ ]:
# Simpan sebagai Numpy Arrays
np.save(os.path.join(PROCESSED_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(PROCESSED_DIR, 'X_test.npy'), X_test_scaled)
np.save(os.path.join(PROCESSED_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(PROCESSED_DIR, 'y_test.npy'), y_test)

# Simpan nama fitur
with open(os.path.join(PROCESSED_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_names, f)

print(f"  Data terproses berhasil disimpan!")
print(f"  Lokasi: {PROCESSED_DIR}")
print(f"\n  File:")
print(f"    - X_train.npy ({X_train_scaled.nbytes / 1024**2:.2f} MB)")
print(f"    - X_test.npy ({X_test_scaled.nbytes / 1024**2:.2f} MB)")
print(f"    - y_train.npy ({y_train.nbytes / 1024**2:.2f} MB)")
print(f"    - y_test.npy ({y_test.nbytes / 1024**2:.2f} MB)")
print(f"    - scaler.pkl")
print(f"    - label_encoder.pkl")
print(f"    - feature_names.json")

  Data terproses berhasil disimpan!
  Lokasi: /content/drive/My Drive/CICIIoT2025/Baseline8Multiclass/Processed_Data/

  File:
    - X_train.npy (209.24 MB)
    - X_test.npy (24.55 MB)
    - y_train.npy (2.95 MB)
    - y_test.npy (0.35 MB)
    - scaler.pkl
    - label_encoder.pkl
    - feature_names.json


###**1.13 Ringkasan Pre-Processing**

In [ ]:
summary = {
    'nama_dataset': 'CICIIoT2025',
    'total_samples_original': 685671,
    'total_samples_cleaned': int(len(X_train) + len(X_test)),
    'samples_removed': 685671 - int(len(X_train) + len(X_test)),
    'removal_percentage': float(((685671 - int(len(X_train) + len(X_test)))/685671)*100),
    'jumlah_fitur': len(feature_names),
    'jumlah_kategori': len(le.classes_),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'train_test_split': 'Time-based (windows 1-8 vs 9-10)',
    'train_windows': train_windows,
    'test_windows': test_windows,
    'normalisasi': 'StandardScaler',
    'label_encoding': 'LabelEncoder',
    'feature_names': feature_names,
    'category_names': le.classes_.tolist(),
    'category_mapping': CATEGORY_MAPPING,
    'imbalance_ratio': float(cat_imbalance_ratio),
    'majority_class': str(category_counts.idxmax()), # Add this line
    'minority_class': str(category_counts.idxmin()), # Add this line
    'data_leakage': 'None - Verified'
}

with open(os.path.join(RESULTS_DIR, 'preprocessing_summary.json'), 'w') as f:
    json.dump(summary, f, indent=4)

print(" Preprocessing selesai!")
print(f"\n  Dataset: {summary['nama_dataset']}")
print(f"  Tipe Klasifikasi: {summary['klasifikasi_type']}")
print(f"  Total Samples (Original): {summary['total_samples_original']:,}")
print(f"  Total Samples (Cleaned): {summary['total_samples_cleaned']:,}")
print(f"  Samples Removed: {summary['samples_removed']:,} ({summary['removal_percentage']:.1f}%)")
print(f"  Jumlah Fitur: {summary['jumlah_fitur']}")
print(f"  Jumlah Kategori: {summary['jumlah_kategori']}")
print(f"  Train Samples: {summary['train_samples']:,}")
print(f"  Test Samples: {summary['test_samples']:,}")
print(f"  Split Strategy: {summary['train_test_split']}")
print(f"  Imbalance Ratio: 1:{summary['imbalance_ratio']:.0f}")
print(f"  Data Leakage: {summary['data_leakage']}")

print("\n" + "="*70)
print("LANGKAH 1 SELESAI - Data Siap untuk Training!")
print("="*70)

 Preprocessing selesai!

  Dataset: CICIIoT2025
  Tipe Klasifikasi: Multiclass_8Categories
  Total Samples (Original): 685,671
  Total Samples (Cleaned): 431,593
  Samples Removed: 254,078 (37.1%)
  Jumlah Fitur: 71
  Jumlah Kategori: 8
  Train Samples: 386,274
  Test Samples: 45,319
  Split Strategy: Time-based (windows 1-8 vs 9-10)
  Imbalance Ratio: 1:39
  Data Leakage: None - Verified

LANGKAH 1 SELESAI - Data Siap untuk Training!


In [ ]:
# Cek RAM Sistem Tersisa
mem = psutil.virtual_memory()
print(f"\n[System Status]")
print(f"  RAM Used: {(mem.total - mem.available) / (1024**3):.2f} GB")
print(f"  RAM Available: {mem.available / (1024**3):.2f} GB")
print(f"  RAM Total: {mem.total / (1024**3):.2f} GB")
print(f"  RAM Usage: {mem.percent:.1f}%")


[System Status]
  RAM Used: 5.15 GB
  RAM Available: 7.52 GB
  RAM Total: 12.67 GB
  RAM Usage: 40.6%
